# Zhihu Article Parser Test

Platform: zhihu / zhuanlan.zhihu.com

Strategy: Playwright + system Chrome (headful) + stealth anti-detection

Output: Markdown (headings, bold, images, formulas preserved)

## 1. Config

In [ ]:
import os
TEST_URL = "https://zhuanlan.zhihu.com/p/2032083217969362749"  # Change to any zhihu article
OUTPUT_DIR = os.path.join(os.getcwd(), "test-output")


## 2. Environment Check

In [ ]:
import os

chrome_paths = [
    r"C:\\Program Files\\Google\\Chrome\\Application\\chrome.exe",
    r"C:\\Program Files (x86)\\Google\\Chrome\\Application\\chrome.exe",
    os.path.expanduser(r"~\\AppData\\Local\\Google\\Chrome\\Application\\chrome.exe"),
]
chrome_found = next((p for p in chrome_paths if os.path.exists(p)), None)
print(f"Chrome: {chrome_found or 'NOT FOUND'}")

for pkg, cmd in [("playwright", "pip install playwright"),
                 ("markdownify", "pip install markdownify"),
                 ("bs4", "pip install beautifulsoup4")]:
    try:
        __import__(pkg)
        print(f"  [OK] {pkg}")
    except ImportError:
        print(f"  [MISSING] {pkg} -> {cmd}")
        raise SystemExit("Install missing packages first.")

## 3. Core Functions (6-layer fallback + Markdown conversion)

In [ ]:
import json
import re
from bs4 import BeautifulSoup
from markdownify import markdownify as md

def extract_zhihu_data(html: str):
    soup = BeautifulSoup(html, "html.parser")
    result = {"title": "", "author": "", "content_html": "", "method": ""}

    # L1: js-initialData
    script = soup.find("script", id="js-initialData")
    if script:
        try:
            data = json.loads(script.string)
            articles = data.get("initialState", {}).get("entities", {}).get("articles", {})
            answers = data.get("initialState", {}).get("entities", {}).get("answers", {})
            items = list(articles.values()) or list(answers.values())
            if items:
                item = items[0]
                result["title"] = item.get("title", "")
                result["author"] = item.get("author", {}).get("name", "")
                result["content_html"] = item.get("content", "")
                result["method"] = "js-initialData"
                return result
        except Exception as e:
            print(f"  L1 error: {e}")

    # L2: window._INITIAL_STATE_
    for s in soup.find_all("script"):
        if s.string and "window._INITIAL_STATE_" in s.string:
            try:
                m = re.search(r"window\._INITIAL_STATE_\s*=\s*({.+?});\s*</script>", str(s), re.DOTALL)
                if m:
                    data = json.loads(m.group(1))
                    articles = data.get("entities", {}).get("articles", {})
                    if articles:
                        item = list(articles.values())[0]
                        result["title"] = item.get("title", "")
                        result["author"] = item.get("author", {}).get("name", "")
                        result["content_html"] = item.get("content", "")
                        result["method"] = "window._INITIAL_STATE_"
                        return result
            except Exception as e:
                print(f"  L2 error: {e}")

    # L3: DOM selector
    container = soup.select_one(".Post-RichTextContainer, .RichContent-inner")
    if container:
        result["content_html"] = str(container)
        result["method"] = "DOM-selector"
        h1 = soup.select_one("h1.Post-Title, .QuestionHeader-title")
        if h1: result["title"] = h1.get_text(strip=True)
        author_link = soup.select_one(".AuthorInfo-name, a.UserLink-link")
        if author_link: result["author"] = author_link.get_text(strip=True)
        return result

    # L4: OG fallback
    og_title = soup.find("meta", property="og:title")
    og_desc = soup.find("meta", property="og:description")
    result["title"] = og_title["content"] if og_title else "Zhihu Content"
    result["content_html"] = f"<p>{og_desc['content']}</p>" if og_desc else ""
    result["method"] = "og-extract"
    return result

def preprocess_zhihu_html(content_html: str) -> str:
    """Pre-process before markdownify (aligns with platform-parser.ts turndown rules)"""
    soup = BeautifulSoup(content_html, "html.parser")
    for figure in soup.find_all("figure"):
        img = figure.find("img")
        caption = figure.find("figcaption")
        if img:
            src = img.get("src") or img.get("data-actualsrc", "")
            cap = caption.get_text(strip=True) if caption else ""
            new_html = f'<p><img src="{src}" alt="{cap}"></p>'
            if cap: new_html += f'<p><em>{cap}</em></p>'
            figure.replace_with(BeautifulSoup(new_html, "html.parser"))
    for noscript in soup.find_all("noscript"):
        img_match = re.search(r'<img[^>]*src="([^"]+)"', str(noscript))
        if img_match:
            noscript.replace_with(BeautifulSoup(f'<img src="{img_match.group(1)}">', "html.parser"))
        else: noscript.decompose()
    for img in soup.find_all("img"):
        actual = img.get("data-actualsrc")
        if actual and not img.get("src"):
            img["src"] = actual
    return str(soup)

def html_to_markdown(content_html: str, title: str = "", author: str = "") -> str:
    processed = preprocess_zhihu_html(content_html)
    markdown = md(processed, heading_style="ATX", bullets="-").strip()
    parts = []
    if title: parts.append(f"# {title}")
    if author: parts.append(f"> Author: {author}")
    parts.append(markdown)
    return "".join(parts)

## 4. Playwright Fetch (headful + stealth)

In [ ]:
from playwright.sync_api import sync_playwright

with sync_playwright() as p:
    launch_opts = {
        "headless": False,
        "args": ["--disable-blink-features=AutomationControlled"],
    }
    if chrome_found: launch_opts["channel"] = "chrome"

    browser = p.chromium.launch(**launch_opts)
    ctx = browser.new_context(
        viewport={"width": 1920, "height": 1080},
        user_agent="Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36",
        locale="zh-CN", timezone_id="Asia/Shanghai",
    )
    ctx.add_init_script("""
        Object.defineProperty(navigator, 'webdriver', { get: () => undefined });
        Object.defineProperty(navigator, 'plugins', { get: () => [1,2,3,4,5] });
        window.chrome = { runtime: {} };
        Object.defineProperty(navigator, 'languages', { get: () => ['zh-CN','zh','en'] });
    """)
    page = ctx.new_page()
    page.goto(TEST_URL, wait_until="domcontentloaded", timeout=30000)
    try:
        page.wait_for_selector(".Post-RichTextContainer, .RichContent-inner", timeout=15000)
    except: pass
    page.wait_for_timeout(3000)
    html = page.content()
    browser.close()

print(f"Fetched HTML: {len(html)} chars")

## 5. Parse & Convert

In [ ]:
data = extract_zhihu_data(html)
markdown = html_to_markdown(data['content_html'], data['title'], data['author'])
plaintext = re.sub(r'<[^>]+>', ' ', data['content_html']).replace("  ", " ").strip()

print(f"Method: {data['method']}")
print(f"Title: {data['title']}")
print(f"Author: {data['author']}")
print(f"HTML content: {len(data['content_html'])} chars")
print(f"Markdown: {len(markdown)} chars")
print(f"Plaintext: {len(plaintext)} chars")

## 6. Preview

In [ ]:
from IPython.display import Markdown as IPMarkdown, display
preview = markdown[:4000] + ("... (truncated)" if len(markdown) > 4000 else "")
display(IPMarkdown(preview))

## 7. Save Results

In [ ]:
import os
from urllib.parse import urlparse

os.makedirs(OUTPUT_DIR, exist_ok=True)
slug = urlparse(TEST_URL).path.strip("/").replace("/", "-") or "page"

paths = {
    "html": os.path.join(OUTPUT_DIR, f"{slug}.html"),
    "md": os.path.join(OUTPUT_DIR, f"{slug}.md"),
    "txt": os.path.join(OUTPUT_DIR, f"{slug}.txt"),
}
with open(paths["html"], "w", encoding="utf-8") as f: f.write(html)
with open(paths["md"], "w", encoding="utf-8") as f: f.write(markdown)
with open(paths["txt"], "w", encoding="utf-8") as f: f.write(plaintext)

for k, p in paths.items():
    print(f"  {k}: {p} ({os.path.getsize(p)} bytes)")